# CarDD YOLO11 fine-tune — config-driven template

Generic Kaggle training template, replacing the old one-notebook-per-experiment approach
(`kaggle_train_yolo11_tuned.ipynb`, `kaggle_train_yolo11_albumentations.ipynb`). The
train/eval logic itself now lives in the `cardd` package (`src/cardd/`), not in this notebook —
a new experiment is a new YAML file under `configs/experiments/`, not a copy-pasted notebook.

Setup: attach `CarDD_COCO` as a dataset input, GPU + Internet on, run top to bottom.

In [ ]:
import torch

print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), "Enable GPU in Notebook Settings (Accelerator: GPU T4 x2 or P100) before continuing."

## Clone the repo and install `cardd`

Pinned to a tag so a run is always reproducible against known code, not a moving `main`. Cloning
(rather than a bare `pip install git+...`) also gives us the `configs/experiments/*.yaml` files on
disk, not just the installed Python package. Deliberately does **not** touch `torch`/`torchvision` -
Kaggle's own preinstalled, GPU-matched build is left as-is (see `pyproject.toml`'s note on this).

In [ ]:
from pathlib import Path

GIT_REF = "v0.1.0"  # tag, branch, or commit - pin to a tag for a reproducible run
REPO_DIR = Path("/kaggle/working/vehicleDD")

if not REPO_DIR.exists():
    !git clone --branch {GIT_REF} --depth 1 https://github.com/JShi12/vehicleDD.git {REPO_DIR}
%cd {REPO_DIR}
!pip install -q -e ".[train]"

import cardd
print("cardd", cardd.__version__)

## Confirm dataset mount path and pick an experiment config

In [ ]:
# <-- edit to match the folder name printed by `!ls /kaggle/input`
KAGGLE_INPUT_DIR = Path("/kaggle/input/cardd-coco")
COCO_ROOT = KAGGLE_INPUT_DIR / "CarDD_COCO" if (KAGGLE_INPUT_DIR / "CarDD_COCO").exists() else KAGGLE_INPUT_DIR
assert (COCO_ROOT / "annotations" / "instances_train2017.json").exists(), (
    f"Can't find CarDD annotations under {COCO_ROOT} - check KAGGLE_INPUT_DIR matches the mount name above"
)

# Swap this to try a different experiment - everything else in this notebook is generic.
# Available: 01_baseline, 02_imgsz1024, 03_tuned, 04_albumentations (see configs/experiments/)
CONFIG = "configs/experiments/03_tuned.yaml"
print("Using COCO_ROOT =", COCO_ROOT)
print("Using CONFIG =", CONFIG)

## COCO -> YOLO conversion

In [ ]:
!cardd-convert --coco-root "{COCO_ROOT}" --out-root data/cardd_yolo --data-yaml configs/cardd_yolo.yaml

## Sanity check
Redraw decoded YOLO boxes to confirm the conversion before training.

In [ ]:
!cardd-verify --data-yaml configs/cardd_yolo.yaml

from IPython.display import Image as IPImage
IPImage(filename="outputs/sanity/yolo_boxes_train2017.png")

## Train
Trains, then runs the held-out test evaluation, then writes `metrics.json` and logs everything to
MLflow (`mlruns/`) - all via the one canonical `cardd.train.run_training` code path. Override any
config value ad hoc with `--set key.subkey=value`, e.g. `--set train.epochs=50`.

In [ ]:
!cardd-train --config {CONFIG}

## Package outputs
Download the zips into the local repo: extract `runs.zip` into `outputs/kaggle_run/<run_name>/`
and `mlruns.zip` into your local `mlruns/` to merge this run's experiment-tracking history in.

In [ ]:
import shutil

run_name = Path(CONFIG).stem  # e.g. "03_tuned"
shutil.make_archive(f"/kaggle/working/{run_name}_runs", "zip", REPO_DIR / "runs")
shutil.make_archive(f"/kaggle/working/{run_name}_mlruns", "zip", REPO_DIR / "mlruns")
print(f"Zipped to /kaggle/working/{run_name}_runs.zip and {run_name}_mlruns.zip")